# TorchLens Facets Tutorial

Facets are named views over captured activations. This notebook is a runnable tour of the current facets API: semantic reads, structural outputs, gradients, interventions, reconstruction, registration, TransformerLens aliases, fallback facets, and patching helpers.

The Hugging Face examples use tiny config-built transformer modules, so no model download is needed. The weights are random; the goal is to make the API easy to inspect and tinker with.


## Setup

Import TorchLens from this checkout, keep the tiny models deterministic, and silence two expected implementation warnings from transformer tracing so the outputs stay readable.


In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path
from typing import Any

import torch
from torch import nn

repo_root = (
    Path.cwd().resolve() if (Path.cwd() / "torchlens").exists() else Path.cwd().resolve().parent
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torchlens as tl  # noqa: E402
from torchlens.semantic import FacetSpec, MissingFacetError, MissingGradient  # noqa: E402

warnings.filterwarnings("ignore", message="TorchLens detected a functorch.*")
warnings.filterwarnings("ignore", message="TorchLens intervention-ready output traversal.*")
warnings.filterwarnings("ignore", message="rerun graph_shape_hash diverged.*")

torch.manual_seed(7)
torch.set_grad_enabled(True)


def describe_tensor(name: str, value: Any) -> None:
    """Print a compact tensor summary for notebook output.

    Parameters
    ----------
    name:
        Label to print.
    value:
        Tensor or tensor-like facet value.

    Returns
    -------
    None
        The summary is printed.
    """

    tensor = value.value if hasattr(value, "value") and torch.is_tensor(value.value) else value
    if torch.is_tensor(tensor):
        print(
            f"{name:<18} shape={tuple(tensor.shape)!s:<16} "
            f"mean={tensor.float().mean().item(): .5f} std={tensor.float().std().item(): .5f}"
        )
    else:
        print(f"{name:<18} {tensor!r}")

## Tiny Real Transformer Models

The main semantic demos use real Hugging Face GPT-2 and DistilBERT classes with small random configs. GPT-2 gives writable `q`/`k`/`v` facets backed by real projection ops; DistilBERT SDPA gives reconstruction facets (`scores`, `pattern`, `z`, `result`).


In [ ]:
from transformers import DistilBertConfig, DistilBertModel, GPT2Config, GPT2LMHeadModel


def make_tiny_gpt2() -> GPT2LMHeadModel:
    """Build a tiny GPT-2 language model from config only.

    Returns
    -------
    GPT2LMHeadModel
        Eval-mode GPT-2 model with one block and four heads.
    """

    config = GPT2Config(
        vocab_size=64,
        n_positions=16,
        n_embd=16,
        n_layer=1,
        n_head=4,
        n_inner=32,
        resid_pdrop=0.0,
        embd_pdrop=0.0,
        attn_pdrop=0.0,
        use_cache=False,
    )
    return GPT2LMHeadModel(config).eval()


def make_tiny_distilbert() -> DistilBertModel:
    """Build a tiny DistilBERT encoder from config only.

    Returns
    -------
    DistilBertModel
        Eval-mode DistilBERT model with one SDPA attention layer.
    """

    config = DistilBertConfig(
        vocab_size=64,
        n_layers=1,
        dim=16,
        hidden_dim=32,
        n_heads=4,
        dropout=0.0,
        attention_dropout=0.0,
    )
    config._attn_implementation = "sdpa"
    return DistilBertModel(config).eval()


gpt2_model = make_tiny_gpt2()
distilbert_model = make_tiny_distilbert()
tokens = torch.tensor([[1, 2, 3, 4]])
corrupted_tokens = torch.tensor([[1, 2, 5, 4]])

print(type(gpt2_model).__name__, sum(p.numel() for p in gpt2_model.parameters()), "parameters")
print(
    type(distilbert_model).__name__,
    sum(p.numel() for p in distilbert_model.parameters()),
    "parameters",
)

## 1. Basic Facet Reads

`module.facets` exposes semantic names from built-in recipes. Attention heads can be sliced with `module.facets.head(i).q`, while MLP recipes expose `up_out`, `intermediate`, `down_out`, and `output`.


In [ ]:
gpt2_log = tl.trace(gpt2_model, tokens, layers_to_save="all", save_arg_values=True)
attn = gpt2_log.modules["transformer.h.0.attn"].facets
mlp = gpt2_log.modules["transformer.h.0.mlp"].facets

print("attention recipe:", attn.recipe_source)
print("attention keys:", attn.keys())
for name in ["q", "k", "v", "attn_out"]:
    describe_tensor(f"attn.{name}", attn[name])
describe_tensor("head(3).q", attn.head(3).q)

print("\nMLP recipe:", mlp.recipe_source)
print("MLP keys:", mlp.keys())
for name in ["up_out", "intermediate", "down_out", "output"]:
    describe_tensor(f"mlp.{name}", mlp[name])

## 2. Structural Outputs as Facets

Structural facets come from captured output containers. Named outputs can be read by name, positional outputs use `out0`, `out1`, and `module.outs` remains the raw structural-output alias.


In [ ]:
class MaxBlock(nn.Module):
    """Module returning a named torch structseq."""

    def forward(self, x: torch.Tensor) -> Any:
        """Return max values and indices.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        Any
            Torch max return object with ``values`` and ``indices``.
        """

        return torch.max(x, dim=1)


class TupleBlock(nn.Module):
    """Module returning a positional tuple."""

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Return two positional tensor outputs.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        tuple[torch.Tensor, torch.Tensor]
            Two derived tensors.
        """

        return x + 10, x - 10


class StructuralModel(nn.Module):
    """Wrapper with named and positional multi-output children."""

    def __init__(self) -> None:
        """Initialize child modules."""

        super().__init__()
        self.named = MaxBlock()
        self.positional = TupleBlock()

    def forward(self, x: torch.Tensor) -> tuple[Any, tuple[torch.Tensor, torch.Tensor]]:
        """Run both structural-output examples.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        tuple[Any, tuple[torch.Tensor, torch.Tensor]]
            Named and positional child outputs.
        """

        return self.named(x), self.positional(x)


struct_x = torch.tensor([[1.0, 5.0, 3.0], [4.0, 2.0, 9.0]])
struct_log = tl.trace(StructuralModel(), struct_x, layers_to_save="all")
named = struct_log.modules["named"].facets
positional_module = struct_log.modules["positional"]
positional = positional_module.facets

print("named keys:", named.keys())
describe_tensor("named.values", named["values"])
describe_tensor("named.indices", named["indices"])
print("\npositional keys:", positional.keys())
describe_tensor("positional.out0", positional["out0"])
describe_tensor("positional.out1", positional["out1"])
print("module.outs alias shapes:", [tuple(out.shape) for out in positional_module.outs])

## 3. Per-Facet Gradients

Default traces do not save gradients, so `facet.grad` returns a `MissingGradient` sentinel. Recapture with `backward_ready=True` and `gradients_to_save="all"`, then call `log_backward(loss)` to read gradients through the same facet transform chain.


In [ ]:
default_grad_log = tl.trace(gpt2_model, tokens, layers_to_save="all")
missing_grad = default_grad_log.modules["transformer.h.0.attn"].facets.head(3).q.grad
print(type(missing_grad).__name__)
print(missing_grad.reason)

grad_log = tl.trace(
    gpt2_model,
    tokens,
    layers_to_save="all",
    backward_ready=True,
    save_grads="all",
)
loss = grad_log[grad_log.output_layers[0]].out[..., 7].sum()
grad_log.log_backward(loss)
q_grad = grad_log.modules["transformer.h.0.attn"].facets.head(3).q.grad
describe_tensor("head(3).q.grad", q_grad)

## 4. Facet Intervention

Facet selectors resolve through the same `FacetSpec` chain. Here a fork zero-ablates one GPT-2 query head and reruns the model; the logits change and the selected facet slice is zero in the edited trace.


In [ ]:
clean_log = tl.trace(gpt2_model, tokens, layers_to_save="all", save_arg_values=True)
clean_logits = clean_log[clean_log.output_layers[0]].out.detach().clone()

ablated = clean_log.fork("zero_q_head_1")
ablated.attach_hooks(tl.head(1, "q"), tl.zero_ablate())
ablated.run(gpt2_model, tokens)
ablated_logits = ablated[ablated.output_layers[0]].out.detach()

print("max |logit delta|:", (ablated_logits - clean_logits).abs().max().item())
print(
    "nonzero entries in ablated head q:",
    int(torch.count_nonzero(ablated.modules["transformer.h.0.attn"].facets.head(1).q).item()),
)

## 5. Attention Reconstruction

With `reconstruction_ready=True`, fused SDPA internals can be reconstructed and validated from saved call arguments. Without that capture mode, the declared reconstruction facets appear as `needs_capture` in the provisional `menu()` and direct access raises `MissingFacetError` with the missing prerequisite.


In [ ]:
recon_log = tl.trace(
    distilbert_model,
    tokens,
    layers_to_save="all",
    save_arg_values=True,
    reconstruction_ready=True,
)
recon_attn = recon_log.modules["transformer.layer.0.attention"].facets
for name in ["scores", "pattern", "z", "result"]:
    describe_tensor(name, recon_attn[name])

pattern_row_sums = recon_attn.pattern.sum(dim=-1)
print(
    "pattern rows sum to 1:",
    torch.allclose(pattern_row_sums, torch.ones_like(pattern_row_sums), atol=1e-5),
)
print(
    "result heads sum to out_lin output:",
    torch.allclose(
        recon_attn.result.sum(dim=-2),
        recon_log.modules["transformer.layer.0.attention.out_lin"].out,
        atol=1e-5,
        rtol=1e-4,
    ),
)

default_recon_log = tl.trace(distilbert_model, tokens, layers_to_save="all")
default_facets = default_recon_log.modules["transformer.layer.0.attention"].facets
print("\ndefault trace pattern status:", default_facets.menu()["pattern"].status)
try:
    default_facets.pattern
except MissingFacetError as exc:
    print(exc)

## 6. Residual-Stream Facets and Per-Head Results

Transformer block recipes expose `resid_pre`, `resid_mid`, and `resid_post`. Reconstructed attention `result` is shaped `[batch, pos, head, d_model]`, so `head(i).result` reads one head contribution.


In [ ]:
block = recon_log.modules["transformer.layer.0"].facets
for name in ["resid_pre", "resid_mid", "resid_post"]:
    describe_tensor(name, block[name])

describe_tensor("head(2).result", recon_attn.head(2).result)
print("all-head result shape:", tuple(recon_attn.result.shape))

## 7. Registration

Recipes return `FacetSpec` objects or ordinary metadata. The decorator records match metadata and declared facet names. The same recipe can be included explicitly with `tl.trace(..., recipes=[...])`, used in a `with tl.facets.using(...)` context, and removed from the process registry with `tl.facets.reset()`.


In [ ]:
tl.facets.reset()


class RecipeDemoModel(nn.Module):
    """Small model for custom facet registration."""

    def __init__(self) -> None:
        """Initialize the projection child."""

        super().__init__()
        self.proj = nn.Linear(4, 3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the projection.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            Projected output.
        """

        return self.proj(x)


@tl.facets.register(
    class_name="Linear",
    target_scope="module",
    facets=("first_feature", "feature_count"),
)
def linear_first_feature(module: Any) -> dict[str, Any]:
    """Expose a selected output feature using the FacetSpec API.

    Parameters
    ----------
    module:
        TorchLens module record for a linear layer.

    Returns
    -------
    dict[str, Any]
        Custom facets for the linear layer.
    """

    output_op = module.trace.ops[module.calls[0].output_ops[0]]
    output_spec = FacetSpec.from_home(output_op, recipe_id="linear_first_feature")
    return {
        "first_feature": output_spec.select(-1, 0),
        "feature_count": int(module.out.shape[-1]),
    }


demo_x = torch.randn(2, 4)
registered_log = tl.trace(RecipeDemoModel(), demo_x, layers_to_save="all")
print("registered keys:", registered_log.modules["proj"].facets.keys())
describe_tensor("first_feature", registered_log.modules["proj"].facets.first_feature)
print("feature_count:", registered_log.modules["proj"].facets.feature_count)
print("info:", tl.facets.info("Linear"))

per_trace_log = tl.trace(
    RecipeDemoModel(), demo_x, layers_to_save="all", recipes=[linear_first_feature]
)
print("per-trace has first_feature:", per_trace_log.modules["proj"].facets.has("first_feature"))

with tl.facets.using(linear_first_feature):
    context_log = tl.trace(RecipeDemoModel(), demo_x, layers_to_save="all")
print("context has first_feature:", context_log.modules["proj"].facets.has("first_feature"))

tl.facets.reset()
after_reset_log = tl.trace(RecipeDemoModel(), demo_x, layers_to_save="all")
print("after reset has first_feature:", after_reset_log.modules["proj"].facets.has("first_feature"))

## 8. TransformerLens Aliases

TransformerLens-style hook names are opt-in aliases. Enable them before capture, then names like `hook_pattern`, `hook_result`, and `hook_resid_pre` resolve to TorchLens native facets.


In [ ]:
tl.facets.enable_transformerlens_aliases(True)
try:
    alias_log = tl.trace(
        distilbert_model,
        tokens,
        layers_to_save="all",
        save_arg_values=True,
        reconstruction_ready=True,
    )
    alias_attn = alias_log.modules["transformer.layer.0.attention"].facets
    alias_block = alias_log.modules["transformer.layer.0"].facets
    print(
        "alias keys present:",
        [name for name in ["hook_pattern", "hook_result"] if name in alias_attn.keys()],
    )
    print("pattern alias equal:", torch.allclose(alias_attn["hook_pattern"], alias_attn.pattern))
    describe_tensor("hook_resid_pre", alias_block["hook_resid_pre"])
finally:
    tl.facets.enable_transformerlens_aliases(False)

## 9. Module-Path Fallback

A plain un-recipe'd module still has structural output facets. This is the fallback path behind `.facets` on arbitrary `nn.Module` records.


In [ ]:
class PlainLeaf(nn.Module):
    """Un-recipe'd module used to show structural fallback."""

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return a simple tensor output.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            Sine of the input.
        """

        return x.sin()


class PlainModel(nn.Module):
    """Wrapper exposing a named plain leaf."""

    def __init__(self) -> None:
        """Initialize child module."""

        super().__init__()
        self.custom = PlainLeaf()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the custom leaf.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            Leaf output.
        """

        return self.custom(x)


plain_x = torch.randn(2, 3)
plain_log = tl.trace(PlainModel(), plain_x, layers_to_save="all")
plain_facets = plain_log.modules["custom"].facets
print("plain recipe source:", plain_facets.recipe_source)
print("plain keys:", plain_facets.keys())
print(
    "fallback equals raw output:", torch.equal(plain_facets["out"], plain_log.modules["custom"].out)
)
describe_tensor("custom.out", plain_facets.out)

## 10. Patching Helpers

The real DistilBERT `result` facet above is reconstructed and read-only, so helper-based patching needs a writable real-tensor facet. This tiny registered attention module exposes `result` as a captured op, then `tl.facets.patching.activation_patch_attention_heads(...)` and `attribution_patch_attention_heads(...)` return `[layer, head]` score tensors.


In [ ]:
tl.facets.reset()


class HeadResult(nn.Module):
    """Return two explicit head-result tensors."""

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Stack one important head and one inert head.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            Tensor shaped ``[batch, pos, head, d_model]``.
        """

        return torch.stack((x, torch.zeros_like(x)), dim=2)


class PatchableAttention(nn.Module):
    """Attention-like module with a writable per-head result facet."""

    def __init__(self) -> None:
        """Initialize result-producing child."""

        super().__init__()
        self.n_heads = 2
        self.result_source = HeadResult()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Sum per-head residual contributions.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            Summed head contribution.
        """

        return self.result_source(x).sum(dim=2)


class PatchableMLP(nn.Module):
    """Small MLP-like module for residual block structure."""

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return a small linear contribution.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            MLP contribution.
        """

        return 0.25 * x


class ToyTransformerBlock(nn.Module):
    """Transformer-like block with attention, MLP, and residual adds."""

    def __init__(self) -> None:
        """Initialize child modules."""

        super().__init__()
        self.attn = PatchableAttention()
        self.mlp = PatchableMLP()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run attention and MLP residual updates.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            Block output.
        """

        resid_mid = x + self.attn(x)
        return resid_mid + self.mlp(resid_mid)


class PatchingToyModel(nn.Module):
    """Wrapper exposing one named transformer block."""

    def __init__(self) -> None:
        """Initialize block."""

        super().__init__()
        self.block = ToyTransformerBlock()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the toy block.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            Model output.
        """

        return self.block(x)


@tl.facets.register(
    class_name="PatchableAttention",
    target_scope="module",
    facets=("result", "attn_out", "n_heads", "head"),
)
def patchable_attention(module: Any) -> dict[str, Any]:
    """Expose writable attention-output facets for the toy module.

    Parameters
    ----------
    module:
        TorchLens module record for ``PatchableAttention``.

    Returns
    -------
    dict[str, Any]
        Facets used by patching helpers.
    """

    result_module = module.trace.modules[f"{module.address}.result_source"]
    result_op = module.trace.ops[result_module.calls[0].output_ops[0]]
    output_op = module.trace.ops[module.calls[0].output_ops[0]]
    return {
        "result": FacetSpec.from_home(result_op, recipe_id="patchable_attention"),
        "attn_out": FacetSpec.from_home(output_op, recipe_id="patchable_attention"),
        "n_heads": 2,
        "head": module.facets.head,
    }


@tl.facets.register(
    class_name="PatchableMLP",
    target_scope="module",
    facets=("output", "up_out"),
)
def patchable_mlp(module: Any) -> dict[str, Any]:
    """Expose writable MLP output facets for the toy module.

    Parameters
    ----------
    module:
        TorchLens module record for ``PatchableMLP``.

    Returns
    -------
    dict[str, Any]
        MLP output facets.
    """

    output_op = module.trace.ops[module.calls[0].output_ops[0]]
    spec = FacetSpec.from_home(output_op, recipe_id="patchable_mlp")
    return {"output": spec, "up_out": spec}


def patch_metric(log: Any) -> torch.Tensor:
    """Return the scalar patching metric.

    Parameters
    ----------
    log:
        TorchLens trace.

    Returns
    -------
    torch.Tensor
        Scalar score from the first token and feature.
    """

    return log[log.output_layers[0]].out[0, 0, 0]


def patch_inputs() -> tuple[torch.Tensor, torch.Tensor]:
    """Return clean and corrupted inputs for patching.

    Returns
    -------
    tuple[torch.Tensor, torch.Tensor]
        Clean and corrupted tensors with gradients enabled.
    """

    clean = torch.zeros(1, 3, 4)
    corrupted = torch.zeros(1, 3, 4)
    clean[0, 0, 0] = 2.0
    return clean.requires_grad_(), corrupted.requires_grad_()


patch_model = PatchingToyModel()
clean_patch_x, corrupted_patch_x = patch_inputs()
activation_scores = tl.facets.patching.activation_patch_attention_heads(
    patch_model,
    clean_patch_x,
    corrupted_patch_x,
    patch_metric,
)
attribution_scores = tl.facets.patching.attribution_patch_attention_heads(
    patch_model,
    clean_patch_x,
    corrupted_patch_x,
    patch_metric,
)

print("activation patch scores [layer, head]:")
print(activation_scores.detach())
print("attribution patch scores [layer, head]:")
print(attribution_scores.detach())


def print_head_heatmap(title: str, scores: torch.Tensor) -> None:
    """Print a compact ASCII heatmap for ``[layer, head]`` patch scores.

    Parameters
    ----------
    title:
        Heatmap title.
    scores:
        Score tensor shaped ``[layer, head]``.

    Returns
    -------
    None
        The heatmap is printed.
    """

    symbols = " .:-=+*#%@"
    values = scores.detach().float().cpu()
    lo = float(values.min().item())
    hi = float(values.max().item())
    span = hi - lo if hi != lo else 1.0
    print(f"\n{title} ASCII heatmap")
    print("      " + " ".join(f"h{head}" for head in range(values.shape[1])))
    for layer in range(values.shape[0]):
        row_symbols = []
        for head in range(values.shape[1]):
            normalized = (float(values[layer, head].item()) - lo) / span
            symbol_index = min(int(normalized * (len(symbols) - 1)), len(symbols) - 1)
            row_symbols.append(symbols[symbol_index])
        row_values = " ".join(f"{float(value):5.2f}" for value in values[layer])
        print(f"L{layer:<3}  " + "  ".join(row_symbols) + f"    {row_values}")


print_head_heatmap("activation", activation_scores)
print_head_heatmap("attribution", attribution_scores)